In [0]:
%run ../../config/utils

In [0]:
from datetime import datetime
import os
import sys
sys.path.append('..')
sys.path.append('../..')


from pyspark import SparkContext
from pyspark import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as sqlf

from lib_cf.cf_io import (
    load_config,
    calculate_filepaths,
)

In [0]:
config_path = dbutils.widgets.get("config_path")
test = False if not dbutils.widgets.get("test") else True
rmse = False if not dbutils.widgets.get("rmse") else True
future = False if not dbutils.widgets.get("future") else True

In [0]:
# (1) ---- ARGUMENTS ---- #

CNF, CFG_PATH = load_config(config_path, test, future, rmse)
PARAMS = dict(list(CNF["shared"].items()) + list(CNF["combine"].items()))
PATHS = CNF["paths"]
OUTPUT_PATH = PATHS["MODEL"]
RUN_NAME = PARAMS["run_name"]
PARAMS, PATHS = calculate_filepaths(PARAMS, PATHS)

now = datetime.now().strftime("%Y%m%d")
columns = ["MBRSHP_SID", "CATEGORY_NAME", "CATEGORY_ID", "prediction"]

In [0]:
cf_data = read_cf_tables(cf_prediction, PARAMS, combined=True)

valid_cf = (
    cf_data.select(*columns)
    .withColumn("valid", sqlf.when(sqlf.col("prediction") > 0, 1).otherwise(0))
    .groupBy("MBRSHP_SID")
    .agg(sqlf.sum(sqlf.col("valid")).alias("count_valid"))
)
invalid_cf = valid_cf.filter(sqlf.col("count_valid") == 0)

CUBE = (
    spark.table(fs_customer_cube_full)
    .filter(
        sqlf.col("FISCAL_WEEK_END").between(PARAMS["start"], PARAMS["end"])
    )
    .join(invalid_cf, on="MBRSHP_SID", how="inner")
)

max_fiscal_week_end = (
    CUBE.groupBy().agg(sqlf.max("FISCAL_WEEK_END")).toPandas().iloc[0, 0]
)

CUBE_sub = (
    CUBE.filter(sqlf.col("FISCAL_WEEK_END") == max_fiscal_week_end)
    .select(
        "MBRSHP_SID",
        "TENURE",
        "LAST_EIGHT_WEEK_SPEND",
        "LAST_TWELVE_WEEK_SPEND",
        "LAST_TWENTY-SIX_WEEK_SPEND",
        "LAST_FIFTY-TWO_WEEK_SPEND",
        "WEEK_TRIPS",
        "LAST_FOUR_WEEK_TRIPS",
        "LAST_EIGHT_WEEK_TRIPS",
        "LAST_TWELVE_WEEK_TRIPS",
        "LAST_TWENTY-SIX_WEEK_TRIPS",
        "LAST_FIFTY-TWO_WEEK_TRIPS",
        "LAST_FISCAL_WEEK_TRIP",
    )
)
invalid_sum = CUBE_sub.summary().drop('MBRSHP_SID')

In [0]:
invalid_sum = invalid_sum.withColumn("RUN_NAME", f.to_date(f.lit(PARAMS["run_name"][-10:]), "yyyy_MM_dd"))

invalid_sum.write.mode('overwrite').option('replaceWhere', f"RUN_NAME = '{PARAMS['run_name'][-10:].replace('_','-')}'").saveAsTable(cf_invalid)